## Reason and Act (ReAct) Pattern

From the article on [Top AI Agentic Workflow Patterns (2026)](https://lekha-bhan88.medium.com/top-ai-agentic-workflow-patterns-that-will-shape-ai-systems-in-2026-736a3141d0e0).

ReAct stands for **Reason + Act**. The core idea is simple: before the model takes any action, it writes out its reasoning — then it acts, observes the result, and reasons again based on what it got back. This loop continues until the model decides it has enough information to produce a final answer.

What makes it different from plain tool use is the explicit reasoning step. With tool use alone, the model just decides "call this tool with these args". With ReAct, the model first explains *why* it's calling that tool, what it expects to learn, and how that fits into solving the broader problem. That chain of thought before each action makes the behavior far more interpretable and often more accurate.

The pattern was introduced in the paper [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629) (Yao et al., 2022). It showed that interleaving reasoning traces with tool calls outperformed both chain-of-thought reasoning alone and tool use alone across a range of tasks.

The loop looks like this:

```
Question
  → Thought: [why I need to do this, what I expect to find]
  → Action: [tool call]
  → Observation: [tool result]
  → Thought: [what this tells me, what to do next]
  → Action: [next tool call]
  → Observation: [result]
  → ...
  → Thought: [I have enough to answer]
  → Final Answer
```

Each iteration is one Thought-Action-Observation triple. You keep going until the model's Thought concludes it's ready to answer.

### How the reasoning trace helps

The Thought step does a few things that aren't obvious at first.

**It surfaces the model's intent before the action.** When you see the model write "I need to look up current interest rates before I can compute the monthly payment", you know exactly what it's doing and why. That's a lot more useful for debugging than a bare `search("interest rates")` call with no context.

**It helps the model avoid premature conclusions.** Models that jump straight to actions tend to anchor on the first result they get. The reasoning step forces the model to evaluate whether it actually has enough information before proceeding, reducing the chance of building an answer on incomplete data.

**It improves multi-step planning within the loop.** Because the Thought is in context for subsequent steps, the model can say things like "the previous search gave me the rate, now I need the principal" — it's not starting from scratch each iteration, it's continuing a reasoning thread.

**It gives you a natural audit trail.** Every decision is logged: what the model thought, what it did, what it got back. This is invaluable when something goes wrong — you can step through the trace and see exactly where the reasoning went off-track.

### Thought-Action-Observation in detail

Each component of the loop has a clear role:

**Thought** — the model's internal monologue. It explains:
- What the current state of the problem is
- What it still needs to know
- Why it's choosing the next action
- What it expects the action to return

This is not the answer — it's the reasoning that justifies the next move.

**Action** — a concrete tool call. It can be anything you've made available: web search, database query, code execution, API call, calculator. The action is determined by the Thought that precedes it.

**Observation** — the raw result from the tool. The model doesn't filter this — you pass back whatever the tool returned. On the next Thought, the model interprets the observation and decides what to do with it.

```
Question: What is the population of the three largest cities in Germany?

Thought: I need to find the three largest German cities by population. I'll search for this.
Action: search("three largest cities in Germany by population")
Observation: Berlin (3.6M), Hamburg (1.9M), Munich (1.5M)

Thought: I have all three cities and their populations. I can now answer directly.
Final Answer: The three largest cities in Germany by population are Berlin (3.6M), 
Hamburg (1.9M), and Munich (1.5M).
```

Short example, but notice the key thing: the Thought before the Final Answer explicitly concludes "I have what I need". The model doesn't just stop — it reasons its way to a stopping condition.

### Use Cases

ReAct is well-suited for any task that requires multiple information-gathering steps before producing an answer.

**Research and question answering** — questions that can't be answered from a single lookup. "What are the tax implications of selling a property in Germany vs France?" requires multiple searches, comparison, and synthesis. The reasoning trace keeps the model oriented across multiple lookups.

**Multi-step data tasks** — "Find all open support tickets from last week where the resolution time exceeded 48 hours and the customer has an enterprise contract." This likely requires several queries — get tickets, filter by date, check resolution times, cross-reference contract type. ReAct handles the composition naturally.

**Debugging and investigation** — "Why is the nightly ETL job failing?" The agent can check logs, inspect schema changes, look up error codes, and cross-reference deployment history. The Thought step shows which hypothesis it's testing at each step.

**Customer-facing agents** — when a user asks something that requires looking up their account, checking product availability, and verifying eligibility before giving an answer. The reasoning trace is useful for support teams reviewing what the agent did.

**Coding assistance with execution** — generate code, run it, observe the output or error, reason about what went wrong, fix and retry. The Thought step prevents the model from thrashing on the same fix repeatedly without understanding why it failed.

### Benefits and Limitations

**Benefits:**

The main advantage over plain tool use is **interpretability**. You can follow the agent's reasoning step by step. This matters a lot when deploying agents in production — if something goes wrong, you need to understand what the agent was thinking, not just what it did.

ReAct also tends to be **more accurate on complex multi-step tasks** than tool use without reasoning. The Thought step forces the model to commit to a hypothesis before acting, which reduces the number of irrelevant tool calls.

Because each Thought is in context for subsequent steps, the model maintains **coherent state across a long sequence of actions** without losing track of the original goal.

**Limitations:**

The reasoning trace increases **token usage** significantly — every step includes a Thought even for actions that don't really need explanation. For simple, single-step tasks this overhead is not worth it.

ReAct is also susceptible to **reasoning errors compounding**. If the Thought at step 2 draws a wrong conclusion from step 1's observation, all subsequent reasoning builds on that mistake. Unlike reflection, there's no built-in mechanism to catch and correct the agent's own reasoning errors mid-loop.

**Hallucinated tool calls** are a known failure mode — the model sometimes writes a Thought that describes a tool call that isn't valid, or passes arguments in a format the tool doesn't accept. Robust argument validation and fallback handling on the tool side is important.

**Infinite loops** are possible if the stopping condition isn't well-defined. Always set a maximum iteration limit. If the model can't reach a final answer within N steps, surface the partial reasoning to the user rather than running forever.

### Implementation Example

In [ ]:
from typing import Any, Callable


# --- Mock tools ---

def search(query: str) -> str:
    """Simulate a web search."""
    mock_results = {
        "population berlin": "Berlin population: 3,645,000 (2023)",
        "population hamburg": "Hamburg population: 1,945,532 (2023)",
        "population munich": "Munich population: 1,512,491 (2023)",
        "largest cities germany": "Largest German cities: 1. Berlin, 2. Hamburg, 3. Munich",
        "gdp germany 2023": "Germany GDP 2023: $4.08 trillion (World Bank)",
    }
    query_lower = query.lower()
    for key, value in mock_results.items():
        if key in query_lower:
            return value
    return f"Search results for '{query}': No specific data found."


def calculator(expression: str) -> str:
    """Evaluate a simple mathematical expression."""
    try:
        result = eval(expression, {"__builtins__": {}})  # restricted eval
        return str(result)
    except Exception as e:
        return f"Calculation error: {e}"


TOOLS: dict[str, Callable] = {
    "search": search,
    "calculator": calculator,
}

print("Tools registered:", list(TOOLS.keys()))

In [ ]:
import re


REACT_SYSTEM_PROMPT = """You are a reasoning agent that solves questions step by step.

For each step, write:
  Thought: [your reasoning about what to do next and why]
  Action: tool_name("argument")

After receiving an Observation, continue with another Thought and Action, or if you
have enough information:
  Thought: [I now have enough to answer]
  Final Answer: [your complete answer]

Available tools:
  - search("query") — search the web for information
  - calculator("expression") — evaluate a math expression

Do not skip the Thought step. Always reason before acting."""


def parse_action(text: str) -> tuple[str, str] | None:
    """
    Extract tool name and argument from an Action line.
    Expects format: Action: tool_name("argument")
    """
    match = re.search(r'Action:\s*(\w+)\("(.+?)"\)', text)
    if match:
        return match.group(1), match.group(2)
    return None


def run_react_agent(
    question: str,
    max_steps: int = 6,
    model: str = "gpt-4o",
) -> dict[str, Any]:
    """
    Run the ReAct loop for a given question.

    Each iteration:
      1. Call the LLM with the current message history
      2. Parse out the Action (if any)
      3. Execute the tool and append the Observation
      4. Repeat until Final Answer or max_steps
    """
    messages = [
        {"role": "system", "content": REACT_SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    trace = []  # full Thought-Action-Observation history for inspection

    for step in range(max_steps):
        print(f"\n--- Step {step + 1} ---")

        # In production: response = openai_client.chat.completions.create(...)
        # For demo, we simulate LLM responses
        llm_output = simulate_llm_response(question, step, messages)

        print(llm_output)
        messages.append({"role": "assistant", "content": llm_output})

        # Check for final answer
        if "Final Answer:" in llm_output:
            final_answer = llm_output.split("Final Answer:")[-1].strip()
            trace.append({"step": step + 1, "type": "final", "content": final_answer})
            print(f"\nFINAL ANSWER: {final_answer}")
            return {"answer": final_answer, "steps": step + 1, "trace": trace}

        # Parse and execute action
        action = parse_action(llm_output)
        if action:
            tool_name, tool_arg = action
            tool_fn = TOOLS.get(tool_name)
            if tool_fn:
                observation = tool_fn(tool_arg)
            else:
                observation = f"Unknown tool: {tool_name}"

            print(f"Observation: {observation}")
            trace.append({"step": step + 1, "type": "action", "tool": tool_name, "arg": tool_arg, "observation": observation})

            # Append observation to conversation so model sees it on the next step
            messages.append({"role": "user", "content": f"Observation: {observation}"})

    return {"answer": "Max steps reached without a final answer.", "steps": max_steps, "trace": trace}


def simulate_llm_response(question: str, step: int, messages: list) -> str:
    """Simulate LLM responses for demo purposes — replace with real LLM call."""
    if step == 0:
        return (
            'Thought: I need to find the three largest German cities first, then get their populations.\n'
            'Action: search("largest cities germany")'
        )
    elif step == 1:
        return (
            'Thought: I know the order is Berlin, Hamburg, Munich. Now I need their populations. Starting with Berlin.\n'
            'Action: search("population berlin")'
        )
    elif step == 2:
        return (
            'Thought: Berlin has 3.645M. Now I need Hamburg.\n'
            'Action: search("population hamburg")'
        )
    elif step == 3:
        return (
            'Thought: Hamburg has 1.945M. Last one is Munich.\n'
            'Action: search("population munich")'
        )
    else:
        return (
            'Thought: I now have all three cities and their populations. I can answer.\n'
            'Final Answer: The three largest cities in Germany by population are:\n'
            '1. Berlin — 3,645,000\n'
            '2. Hamburg — 1,945,532\n'
            '3. Munich — 1,512,491'
        )


result = run_react_agent("What are the populations of the three largest cities in Germany?")
print(f"\nCompleted in {result['steps']} steps")

### Working Implementation with Real LLM Calls

The above example uses simulated responses. This section shows how to wire it up to a real LLM using OpenAI's function-calling API, which makes action parsing more reliable than regex on free-form text.

**Setup**: set `OPENAI_API_KEY` in your environment before running.

In [ ]:
import os
from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from environment

TOOL_DEFINITIONS = [
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search the web for factual information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a Python math expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "A Python math expression"}
                },
                "required": ["expression"]
            }
        }
    }
]

REACT_SYSTEM_PROMPT_REAL = """You are a reasoning agent. Before every action, write a
Thought explaining what you know so far and why you're taking the next step.
This reasoning trace is important — do not skip it."""


def run_react_agent_real(question: str, max_steps: int = 8) -> str:
    """
    ReAct loop using OpenAI function calling.
    The model's tool calls are structured, so no regex parsing needed.
    """
    import json

    messages = [
        {"role": "system", "content": REACT_SYSTEM_PROMPT_REAL},
        {"role": "user", "content": question},
    ]

    for step in range(max_steps):
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=TOOL_DEFINITIONS,
            tool_choice="auto",
        )

        message = response.choices[0].message
        messages.append(message)

        # Print reasoning trace if present
        if message.content:
            print(f"Thought: {message.content}")

        # No tool call means the model is done
        if not message.tool_calls:
            return message.content or "No answer produced."

        # Execute each tool call and feed back observations
        for tool_call in message.tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)

            print(f"Action: {fn_name}({fn_args})")

            tool_fn = TOOLS.get(fn_name)
            observation = tool_fn(**fn_args) if tool_fn else f"Unknown tool: {fn_name}"

            print(f"Observation: {observation}\n")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": observation,
            })

    return "Max steps reached."


# Requires OPENAI_API_KEY
answer = run_react_agent_real("What is the combined population of Berlin and Hamburg?")
print(f"\nFinal Answer: {answer}")

### Variations Worth Knowing

**ReAct with memory** — the agent maintains a separate memory store (key-value or vector) that it can write to and read from across steps. Useful when the number of steps is large enough that relevant observations would otherwise scroll out of the context window.

```python
# Agent writes key facts to memory during the loop
memory.store("berlin_population", "3,645,000")
# Retrieves them later without re-searching
berlin_pop = memory.retrieve("berlin_population")
```

**ReAct with reflection** — after a failed action (tool error, empty result), insert a Reflect step before continuing. The model reasons about why the action failed and how to adjust. This combines the interpretability of ReAct with the error-recovery ability of the Reflection pattern.

```
Thought: I'll search for the current CEO of Acme Corp.
Action: search("Acme Corp CEO 2024")
Observation: No results found.
Reflect: The search returned nothing. I should try a broader query or check the company's official site.
Thought: Let me try searching their website directly.
Action: search("site:acmecorp.com leadership team")
```

**ReAct with human-in-the-loop** — pause before high-stakes actions (sending an email, modifying a record) and surface the Thought + proposed Action to a human for approval before executing.

**Self-ask with search** — a simplified ReAct variant where the model decomposes the question into sub-questions and answers each one in sequence. No general-purpose tool loop — just a structured decompose-search-synthesize pattern. Easier to implement but less flexible.

### How It Fits with Other Patterns

| Pattern | ReAct Relationship |
|---------|-------------------|
| **Tool Use** | ReAct extends tool use by adding explicit reasoning before each action |
| **Reflection** | Reflection improves a single output; ReAct reasons mid-loop while gathering information |
| **Planning** | Planning produces a full plan upfront; ReAct decides each step dynamically based on what it observes |
| **Multi-Agent** | Individual agents in a multi-agent system can internally use ReAct as their decision loop |
| **Adaptive Workflow** | Adaptive workflows can use ReAct as the decision mechanism for choosing which agent to invoke next |

The most important comparison is ReAct vs Planning. Both involve thinking before acting, but the distinction is:
- **Planning**: reason once, produce a full plan, execute it. Good when the task is well-defined and the steps are predictable.
- **ReAct**: reason at each step based on observed results. Good when the next step depends on what you find in the previous one — you can't plan ahead because you don't know what the tool will return.

In practice, you often want both: a planner to produce the high-level task decomposition, and ReAct within each sub-task to handle the actual information gathering adaptively.

### Workflow Diagram

```
┌─────────────────────────────────────────────────────────────┐
│                    USER QUESTION                            │
│    "What is the combined population of Berlin & Hamburg?"   │
└─────────────────────┬───────────────────────────────────────┘
                      │
                      ▼
┌─────────────────────────────────────────────────────────────┐
│                   LLM (ReAct Loop)                          │
│                                                             │
│  Thought: I need Berlin's population first.                 │
│  Action: search("population berlin")                        │
└──────────────────────────┬──────────────────────────────────┘
                           │
                           ▼
              ┌────────────────────────┐
              │       TOOL             │
              │  search("population    │
              │         berlin")       │
              └────────────┬───────────┘
                           │
                           ▼
              Observation: Berlin: 3,645,000
                           │
                           ▼
┌─────────────────────────────────────────────────────────────┐
│  Thought: Got Berlin. Now Hamburg.                          │
│  Action: search("population hamburg")                       │
└──────────────────────────┬──────────────────────────────────┘
                           │
                           ▼
              ┌────────────────────────┐
              │       TOOL             │
              │  search("population    │
              │         hamburg")      │
              └────────────┬───────────┘
                           │
                           ▼
              Observation: Hamburg: 1,945,532
                           │
                           ▼
┌─────────────────────────────────────────────────────────────┐
│  Thought: I have both. Add them: 3,645,000 + 1,945,532      │
│  Action: calculator("3645000 + 1945532")                    │
└──────────────────────────┬──────────────────────────────────┘
                           │
                           ▼
              Observation: 5590532
                           │
                           ▼
┌─────────────────────────────────────────────────────────────┐
│  Thought: I have the answer.                                │
│  Final Answer: Combined population is 5,590,532.            │
└─────────────────────────────────────────────────────────────┘
```

### Summary

ReAct is the go-to pattern for tasks that require multiple tool calls where each step depends on the previous result. The core loop is straightforward: Thought → Action → Observation, repeat until done.

The reasoning trace is what distinguishes it from plain tool use — it makes the agent's behavior interpretable, helps the model stay on track across many steps, and gives you a natural audit trail for debugging.

Key decisions when building a ReAct agent:
- Free-form Thought + Action parsing, or structured function calling? (Function calling is more reliable in production.)
- What is the stopping condition? (Tool call absent, or explicit final-answer token.)
- How many max steps? (Set this; otherwise loops are possible.)
- Should failed actions trigger a Reflect step before continuing?

### Papers

- **ReAct** (Yao et al., 2022) — the original paper introducing the Thought-Action-Observation loop
- **Toolformer** (Schick et al., 2023) — teaching models to use tools via self-supervised learning
- **HotpotQA** — multi-hop QA benchmark commonly used to evaluate ReAct agents
- **Self-Ask** (Press et al., 2022) — decompose-and-search variant of the same idea